# GRUEncoder Training — SOLUSDT Phase A (h32 variant)

Trains the GRUEncoder (hidden=32, dropout=0.2, 5K params) on processed
feature data using a free T4 GPU. This is the reduced-capacity variant
after the diagnostic showed hidden=64 generalizes poorly.

## Prerequisites
1. Upload the project zip to Google Drive:
   - Create `ModelProject` folder in your Drive root
   - Upload `project.tar.gz` there (see cell 4 for how to create it)
2. Run cells in order below.

After training, the best checkpoint (`best.pt`) and metrics are saved back to Drive.

In [ ]:
# Cell 1: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted at /content/drive')

In [ ]:
# Cell 2: Verify GPU
import torch
print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')
else:
    raise RuntimeError('GPU not available — go to Runtime > Change runtime type > T4 GPU')

In [ ]:
# Cell 3: Install dependencies
!pip install --quiet pandas pyarrow pyyaml httpx numba
print('Dependencies installed')

## Prepare the Project

Upload two files to `MyDrive/ModelProject/` in Google Drive:

**1. Project code** (excludes data — much smaller):
```
tar -czf project.tar.gz --exclude='.git' --exclude='__pycache__' --exclude='.venv' --exclude='venv' --exclude='node_modules' --exclude='.pytest_cache' --exclude='data/processed' --exclude='data/raw' .
```
**2. Processed feature data:**
```
tar -czf data_processed.tar.gz -C data/processed/v1/SOLUSDT/1m .
```

Upload both to `MyDrive/ModelProject/` then run cells below.

In [ ]:
# Cell 4: Extract project and data from Drive
import os
import shutil

PROJECT_TAR = '/content/drive/MyDrive/ModelProject/project.tar.gz'
DATA_TAR = '/content/drive/MyDrive/ModelProject/data_processed.tar.gz'
OUT_DIR = '/content/ModelProject'

if not os.path.exists(PROJECT_TAR):
    raise FileNotFoundError(f'Not found: {PROJECT_TAR}')
if not os.path.exists(DATA_TAR):
    raise FileNotFoundError(f'Not found: {DATA_TAR}')

os.makedirs(OUT_DIR, exist_ok=True)
print('Extracting project code...')
!tar -xzf "$PROJECT_TAR" -C "$OUT_DIR"

print('Extracting processed data...')
data_dir = os.path.join(OUT_DIR, 'data', 'processed', 'v1', 'SOLUSDT', '1m')
os.makedirs(data_dir, exist_ok=True)
!tar -xzf "$DATA_TAR" -C "$data_dir"

%cd "$OUT_DIR"
print(f'\nContents of {OUT_DIR}: {os.listdir(".")}')
print(f'Data files in {data_dir}: {len(os.listdir(data_dir))}')

In [ ]:
# Cell 5: Smoke test only (~10s)
# Run a 2-epoch smoke test to verify everything works before the long run.
# This does NOT run full training — that's Cell 6.
smoke_ok = False
try:
    !python -m model.train --config configs/phase_a_gru_h32.yaml --smoke-test-only
    smoke_ok = True
    print('\nSmoke test PASSED — ready for full training.')
except Exception as e:
    print(f'Smoke test FAILED: {e}')

In [ ]:
# Cell 6: Full training run
# This takes ~3-5 minutes on a T4 GPU (vs ~2h on CPU)
if smoke_ok:
    !python -m model.train --config configs/phase_a_gru_h32.yaml
else:
    print('Skipping full training — smoke test did not pass. Fix errors first.')

In [ ]:
# Cell 7: Save checkpoint to Drive
import glob
import json
from pathlib import Path

# Find the latest run
runs_dir = Path(OUT_DIR) / 'model' / 'runs'
run_dirs = sorted(runs_dir.iterdir())
if not run_dirs:
    raise RuntimeError('No training runs found')

# Get the most recent non-smoketest run
real_runs = [d for d in run_dirs if d.is_dir() and 'smoketest' not in d.name]
if not real_runs:
    real_runs = [d for d in run_dirs if d.is_dir()]
latest_run = real_runs[-1].name
print(f'Latest run: {latest_run}')

# Save best checkpoint to Drive
best_path = runs_dir / latest_run / 'checkpoints' / 'best.pt'
if best_path.exists():
    drive_best = f'/content/drive/MyDrive/ModelProject/{latest_run}_best.pt'
    shutil.copy(best_path, drive_best)
    print(f'Saved best.pt to Drive: {drive_best}')
else:
    print('No best.pt found')

# Save metrics to Drive
metrics_path = runs_dir / latest_run / 'metrics.jsonl'
if metrics_path.exists():
    drive_metrics = f'/content/drive/MyDrive/ModelProject/{latest_run}_metrics.jsonl'
    shutil.copy(metrics_path, drive_metrics)
    print(f'Saved metrics to Drive: {drive_metrics}')

# Print final metrics for a quick check
if metrics_path.exists():
    with open(metrics_path) as f:
        lines = f.readlines()
    last = json.loads(lines[-1])
    print(f'\nFinal epoch: {last.get("epoch", "?")}')
    print(f'Val loss:   {last.get("loss", "?")}')
    print(f'NLL:        {last.get("nll", "?")}')
    print(f'Baseline:   {last.get("baseline_loss", "?")}')
    print(f'DeltaBL:    {last.get("baseline_delta", "?")}')

In [ ]:
# Cell 8: Download checkpoint locally
from google.colab import files

best_path = runs_dir / latest_run / 'checkpoints' / 'best.pt'
if best_path.exists():
    files.download(str(best_path))
    print('Downloading best.pt to your local machine...')
else:
    print('No best.pt to download')